# Rabbit — Engine Dashboard

Dark cockpit UI: chat, model stats, expert cache, expert activity.

**Not** Colibrì's UI — Rabbit's own dashboard.

[Open in Colab](https://colab.research.google.com/github/Stanley03-arch/Rabbit/blob/main/notebooks/rabbit_colab_dashboard.ipynb)

## 1. Clone & build

In [ ]:
import os
os.chdir("/content")
!rm -rf Rabbit
!git clone --depth 1 https://github.com/Stanley03-arch/Rabbit.git
os.chdir("/content/Rabbit")
!make clean && make 2>&1 | tail -20
!chmod +x rabbit tools/create_test_model 2>/dev/null; true
!make test-model
!./rabbit version
!ls -la web/

## 2. Install Flask & start dashboard

Colab cannot open `http://127.0.0.1:7860` from your laptop directly.
Use **iframe** proxy (recommended) or the link Colab prints.

In [ ]:
!pip install -q flask

import subprocess, time, sys
from google.colab import output

# Stop any previous server on 7860
subprocess.run("fuser -k 7860/tcp 2>/dev/null", shell=True)
time.sleep(1)

# Start Flask in the background
proc = subprocess.Popen(
    [sys.executable, "tools/rabbit_dashboard.py",
     "--model", "/content/Rabbit/models/tiny-test.rabbit",
     "--host", "0.0.0.0", "--port", "7860"],
    cwd="/content/Rabbit",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait until server is up
for i in range(30):
    line = proc.stdout.readline() if proc.stdout else ""
    if line:
        print(line.rstrip())
    if proc.poll() is not None:
        rest = proc.stdout.read() if proc.stdout else ""
        print(rest)
        raise RuntimeError("dashboard exited early")
    # probe
    try:
        import urllib.request
        urllib.request.urlopen("http://127.0.0.1:7860/", timeout=1)
        print("Server is up.")
        break
    except Exception:
        time.sleep(0.5)
else:
    print("Still starting… opening proxy anyway.")

# Recommended Colab proxy (iframe)
output.serve_kernel_port_as_iframe(7860)
print("\nIf the iframe is blank, run the next cell for a clickable proxy link.")

## 3. Alternative: proxy URL (if iframe is blank)

In [ ]:
from google.colab import output
print("Open this proxy URL:")
print(output.eval_js("google.colab.kernel.proxyPort(7860)"))

### What you should see

- Left: model stats, cache hit rate, expert grid
- Right: chat box
- After Send: reply + updated cache numbers

Synthetic output can look like noise/question marks — engine path is still working.